# HC3 Train / Validation / Test Split

## Load cleaned dataset

In [29]:
import pandas as pd

df = pd.read_csv("../data/processed/expanded_dataset.csv")

## Analyze the cleaned dataset output

In [30]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nNumber of answer rows:", len(df))
print("Number of unique questions:", df["id"].nunique())
print("Repeated question IDs:", df["id"].duplicated().sum())

df.head(10)

Shape: (84066, 6)

Columns:
['id', 'question', 'source', 'text', 'label', 'target']

Number of answer rows: 84066
Number of unique questions: 23867
Repeated question IDs: 60199


,id,question,source,text,label,target
0,0,"Why is every book I hear about a "" NY Times # ...",reddit_eli5,"Basically there are many categories of "" Best ...",human,0
1,0,"Why is every book I hear about a "" NY Times # ...",reddit_eli5,"If you 're hearing about it , it 's because it...",human,0
2,0,"Why is every book I hear about a "" NY Times # ...",reddit_eli5,"One reason is lots of catagories . However , h...",human,0
3,1,"If salt is so bad for cars , why do we use it ...",reddit_eli5,salt is good for not dying in car crashes and ...,human,0
4,1,"If salt is so bad for cars , why do we use it ...",reddit_eli5,"In Minnesota and North Dakota , they tend to u...",human,0
5,1,"If salt is so bad for cars , why do we use it ...",reddit_eli5,"Used to work in the salt industry ( yes , it '...",human,0
6,2,Why do we still have SD TV channels when HD lo...,reddit_eli5,The way it works is that old TV stations got a...,human,0
7,2,Why do we still have SD TV channels when HD lo...,reddit_eli5,HD does n't look like anything at all on an SD...,human,0
8,2,Why do we still have SD TV channels when HD lo...,reddit_eli5,There are a few reasons why SD channels still ...,human,0
9,3,Why has nobody assassinated Kim Jong - un He i...,reddit_eli5,You ca n't just go around assassinating the le...,human,0


## Check class distribution

In [31]:
print("Overall label counts:")
print(df["label"].value_counts())

print("\nOverall label percentages:")
print((df["label"].value_counts(normalize=True) * 100).round(2))

Overall label counts:
label
human      57181
chatgpt    26885
Name: count, dtype: int64

Overall label percentages:
label
human      68.02
chatgpt    31.98
Name: proportion, dtype: float64


### Check balance by source

In [32]:
print(pd.crosstab(df["source"], df["label"]))

print("\nPercent within each source:")
print(
    pd.crosstab(
        df["source"],
        df["label"],
        normalize="index"
    ).mul(100).round(2)
)

label        chatgpt  human
source                     
finance         4503   3933
medicine        1334   1248
open_qa         3546   1187
reddit_eli5    16660  49971
wiki_csai        842    842

Percent within each source:
label        chatgpt  human
source                     
finance        53.38  46.62
medicine       51.67  48.33
open_qa        74.92  25.08
reddit_eli5    25.00  75.00
wiki_csai      50.00  50.00


## Train, Validation, and Test Split

Although the expanded dataset contains one row per answer, all answers that
belong to the same question must remain in the same split.

A question-level table is created using the original question `id`. Splitting
this table first prevents related human and ChatGPT answers from the same
question from appearing across training, validation, and test sets.

### Prepare a question-level table for splitting

In [33]:
questions = (
    df[["id", "source"]]
    .drop_duplicates(subset="id")
    .reset_index(drop=True)
)

print("Unique questions:", len(questions))

questions.head()

Unique questions: 23867


,id,source
0,0,reddit_eli5
1,1,reddit_eli5
2,2,reddit_eli5
3,3,reddit_eli5
4,4,reddit_eli5


In [34]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

In [35]:
train_questions, temp_questions = train_test_split(
    questions,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=questions["source"]
)

validation_questions, test_questions = train_test_split(
    temp_questions,
    test_size=0.5,
    random_state=RANDOM_STATE,
    stratify=temp_questions["source"]
)

### Check the split sizes

In [36]:
print("Question-level split sizes:")

print("Train:", len(train_questions))
print("Validation:", len(validation_questions))
print("Test:", len(test_questions))

print(
    "\nTotal:",
    len(train_questions)
    + len(validation_questions)
    + len(test_questions)
)

Question-level split sizes:
Train: 19093
Validation: 2387
Test: 2387

Total: 23867


In [37]:
total_questions = len(questions)

print(
    "\nTrain %:",
    round(len(train_questions) / total_questions * 100, 2)
)

print(
    "Validation %:",
    round(len(validation_questions) / total_questions * 100, 2)
)

print(
    "Test %:",
    round(len(test_questions) / total_questions * 100, 2)
)


Train %: 80.0
Validation %: 10.0
Test %: 10.0


### Map the question assignments back to answer rows

In [40]:
train_ids = set(train_questions["id"])
validation_ids = set(validation_questions["id"])
test_ids = set(test_questions["id"])

train = df[df["id"].isin(train_ids)].copy()

validation = df[
    df["id"].isin(validation_ids)
].copy()

test = df[df["id"].isin(test_ids)].copy()

### Check answer-level split sizes

In [41]:
print("Answer-level split sizes:")

print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

print(
    "\nTotal:",
    len(train)
    + len(validation)
    + len(test)
)

print("Original dataset:", len(df))

Answer-level split sizes:
Train: 67232
Validation: 8417
Test: 8417

Total: 84066
Original dataset: 84066


### Check for question leakage

In [43]:
train_id_set = set(train["id"])
validation_id_set = set(validation["id"])
test_id_set = set(test["id"])

print(
    "Train / validation overlap:",
    len(train_id_set & validation_id_set)
)

print(
    "Train / test overlap:",
    len(train_id_set & test_id_set)
)

print(
    "Validation / test overlap:",
    len(validation_id_set & test_id_set)
)

Train / validation overlap: 0
Train / test overlap: 0
Validation / test overlap: 0


## Summary of the three splits

In [44]:
def split_summary(name, data):
    print(f"\n{name.upper()}")
    print("-" * 40)

    print("Answer rows:", len(data))
    print("Unique questions:", data["id"].nunique())

    print("\nLabel counts:")
    print(data["label"].value_counts())

    print("\nLabel percentages:")
    print(
        (
            data["label"]
            .value_counts(normalize=True)
            * 100
        ).round(2)
    )

    print("\nSource percentages:")
    print(
        (
            data["source"]
            .value_counts(normalize=True)
            * 100
        ).round(2)
    )

    print("\nSource x label:")
    print(
        pd.crosstab(
            data["source"],
            data["label"]
        )
    )

## Split Summary

The expanded HC3 dataset was divided into training, validation, and test sets using an approximately 80/10/10 split.

- Splitting was performed at the question level using the original question IDs.
- Source/domain distributions were preserved using stratified sampling.
- A random state of 42 makes the split reproducible.
- No question IDs overlap between the training, validation, and test sets.
- All answer rows associated with the same question remain together in a single split.
- The original class distributions were preserved in the training, validation, and test sets.
- Class imbalance will be handled during model training rather than by removing examples from the dataset.

These splits are ready for TF-IDF feature engineering, model training, evaluation, per-domain analysis, and later robustness experiments.

In [47]:
split_summary("Train", train)
split_summary("Validation", validation)
split_summary("Test", test)


TRAIN
----------------------------------------
Answer rows: 67232
Unique questions: 19093

Label counts:
label
human      45743
chatgpt    21489
Name: count, dtype: int64

Label percentages:
label
human      68.04
chatgpt    31.96
Name: proportion, dtype: float64

Source percentages:
source
reddit_eli5    79.28
finance        10.01
open_qa         5.64
medicine        3.06
wiki_csai       2.00
Name: proportion, dtype: float64

Source x label:
label        chatgpt  human
source                     
finance         3587   3146
medicine        1061    998
open_qa         2839    950
reddit_eli5    13328  39975
wiki_csai        674    674

VALIDATION
----------------------------------------
Answer rows: 8417
Unique questions: 2387

Label counts:
label
human      5719
chatgpt    2698
Name: count, dtype: int64

Label percentages:
label
human      67.95
chatgpt    32.05
Name: proportion, dtype: float64

Source percentages:
source
reddit_eli5    79.17
finance        10.10
open_qa         5.63

## Saving the splits

In [49]:
train.to_csv("../data/processed/train.csv", index=False)
validation.to_csv("../data/processed/validation.csv", index=False)
test.to_csv("../data/processed/test.csv", index=False)

print("Saved:")
print("../data/processed/train.csv")
print("../data/processed/validation.csv")
print("../data/processed/test.csv")

Saved:
../data/processed/train.csv
../data/processed/validation.csv
../data/processed/test.csv
